# 06 · Alerta de risco de crédito

Este notebook roda quando a **taxa de inadimplência** calculada pela gold passa do limite definido pelo negócio (lab da aula 17). Ele grava em `growdev.gold.alertas_risco` os segmentos de clientes com mais inadimplência, para a área de crédito agir.

In [ ]:
dbutils.widgets.text("catalogo", "growdev", "Catálogo")
dbutils.widgets.text("job_run_id", "manual", "Id da execução do job")

CATALOGO = dbutils.widgets.get("catalogo")
JOB_RUN_ID = dbutils.widgets.get("job_run_id")

taxa = dbutils.jobs.taskValues.get(
    taskKey="gold", key="taxa_inadimplencia_pct", default=-1, debugValue=-1
)
print(f"Taxa de inadimplência recebida da tarefa gold: {taxa}%")

In [ ]:
from pyspark.sql import functions as F

df_alerta = (
    spark.table(f"{CATALOGO}.gold.resumo_clientes")
    .groupBy("state", "customer_segment")
    .agg(
        F.sum("qtd_emprestimos").alias("emprestimos"),
        F.sum("qtd_inadimplencias").alias("inadimplencias"),
    )
    .withColumn("taxa_pct", F.round(100 * F.col("inadimplencias") / F.col("emprestimos"), 2))
    .orderBy(F.desc("taxa_pct"))
    .limit(10)
    .withColumn("taxa_carteira_pct", F.lit(float(taxa)))
    .withColumn("job_run_id", F.lit(JOB_RUN_ID))
    .withColumn("registrado_em", F.current_timestamp())
)

df_alerta.write.mode("append").saveAsTable(f"{CATALOGO}.gold.alertas_risco")
display(df_alerta)